# Cortex Discovery Pipeline — Step-by-Step

Run each cell to step through the pipeline. Edit prompts and parameters inline to tune the output.

In [1]:
import sys, os, json, logging, re
from pathlib import Path
from collections import Counter

backend_dir = Path("backend").resolve()
sys.path.insert(0, str(backend_dir))

from dotenv import load_dotenv
load_dotenv(Path(".").resolve() / ".env")

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from models import RawItem
from fetchers import FETCHERS, extract_content

print(f"Backend dir: {backend_dir}")
print(f"MINIMAX_API_KEY set: {bool(os.getenv('MINIMAX_API_KEY'))}")
print(f"EXA_API_KEY set: {bool(os.getenv('EXA_API_KEY'))}")
print(f"\nRegistered fetchers: {list(FETCHERS.keys())}")

Backend dir: /Users/philipp/coden/agent_system_design/projects/Hackathon/workspace/backend
MINIMAX_API_KEY set: True
EXA_API_KEY set: True

Registered fetchers: ['reddit', 'hackernews', 'rss', 'arxiv', 'twitter', 'twitter_accounts', 'podcast', 'youtube', 'pubmed', 'exa', 'substack', 'blog', 'newsletter']


## Step 1: Config

Edit this dict to change what gets fetched. This is the single source of truth for the pipeline.

In [2]:
CONFIG = {
    "name": "AI Agents Weekly",
    "description": "What happened in AI agents this week",

    "interests": {
        "field": "AI agents & agentic systems",
        "topics": [
            "AI agent frameworks",
            "tool use and function calling",
            "MCP (Model Context Protocol)",
            "multi-agent orchestration",
            "agent memory and planning",
            "code generation agents",
            "autonomous coding (Devin, Cursor, Claude Code)",
        ],
        "level": "advanced",
        "depth": "technical",
    },

    "discovery": {
        "timeframe_hours": 168,
        "max_topics": 10,
        "relevance_threshold": 0.3,
        "n_clusters": 12,
    },

    "sources": {
        "reddit": {
            "enabled": True,
            "subreddits": ["MachineLearning", "LocalLLaMA", "ClaudeAI", "ChatGPTPro"],
            "min_score": 30,
            "max_results": 100,
            "hours_back": 168,
        },
        "hackernews": {
            "enabled": True,
            "filter": "AI agents OR MCP OR Claude OR agentic OR multi-agent OR tool use OR function calling",
            "min_points": 20,
            "max_results": 30,
            "hours_back": 168,
        },
        "arxiv": {
            "enabled": True,
            "categories": ["cs.AI", "cs.CL", "cs.SE"],
            "max_results": 30,
            "hours_back": 168,
        },
        "rss": {
            "enabled": True,
            "feeds": {
                "openai": "https://openai.com/blog/rss/",
                "anthropic": "https://www.anthropic.com/feed.xml",
                "google_ai": "https://blog.google/technology/ai/rss/",
                "deepmind": "https://deepmind.google/blog/rss.xml",
            },
            "hours_back": 168,
        },
        "substack": {
            "enabled": True,
            "publications": {
                "import_ai": "https://importai.substack.com",
                "ahead_of_ai": "https://magazine.sebastianraschka.com",
                "interconnects": "https://www.interconnects.ai",
                "latent_space": "https://www.latent.space",
            },
            "hours_back": 168,
        },
        "blogs": {
            "enabled": True,
            "feeds": {
                "simon_willison": "https://simonwillison.net/atom/everything/",
                "huggingface": "https://huggingface.co/blog/feed.xml",
                "langchain": "https://blog.langchain.dev/rss/",
                "llamaindex": "https://www.llamaindex.ai/blog/rss.xml",
            },
            "hours_back": 168,
        },
        "exa": {
            "enabled": True,
            "query": "AI agents frameworks tool use MCP agentic coding",
            "category": "news",
            "max_results": 20,
            "hours_back": 168,
        },
        "newsletters": {
            "enabled": True,
            "newsletters": [
                {"name": "TLDR AI", "type": "tldr", "section": "ai"},
                {"name": "The Batch", "type": "deeplearning_batch"},
            ],
            "hours_back": 168,
        },
        "pubmed": {
            "enabled": False,
            "query": "large language model agents",
            "max_results": 10,
            "hours_back": 168,
        },
        "twitter": {
            "enabled": False,
            "keywords": ["AI agents", "MCP", "Claude Code", "agentic"],
            "lang": "en",
        },
        "twitter_accounts": {
            "enabled": False,
            "accounts": ["AnthropicAI", "OpenAI", "LangChainAI", "llaboratory"],
        },
        "podcasts": {
            "enabled": False,
            "shows": [
                {"name": "Latent Space", "feed_url": None},
                {"name": "Gradient Dissent", "feed_url": None},
            ],
            "hours_back": 168,
        },
        "youtube": {
            "enabled": False,
            "channels": {
                "Two Minute Papers": "UCbfYPyITQ-7l4upoX8nvctg",
                "Yannic Kilcher": "UCZHmQk67mSJgfCCTn7xBfew",
            },
            "max_results": 10,
            "hours_back": 168,
        },
    },
}

# --- Normalize config into the flat profile format the pipeline expects ---
from discovery import _normalize_config

profile = _normalize_config(CONFIG)
source_configs = {s["id"]: s for s in profile["sources"]}

print(f"Config: {CONFIG['name']}")
print(f"Field: {profile['field']}")
print(f"Topics: {profile['focus_areas']}")
print(f"Threshold: {profile['relevance_threshold']}, Clusters: {profile['n_clusters']}")
print(f"\nSources:")
for s in profile["sources"]:
    status = "ON" if s.get("enabled", True) else "off"
    print(f"  [{status:>3}] {s['id']} ({s['type']})")

Config: AI Agents Weekly
Field: AI agents & agentic systems
Topics: ['AI agent frameworks', 'tool use and function calling', 'MCP (Model Context Protocol)', 'multi-agent orchestration', 'agent memory and planning', 'code generation agents', 'autonomous coding (Devin, Cursor, Claude Code)']
Threshold: 0.3, Clusters: 12

Sources:
  [ ON] reddit (reddit)
  [ ON] hackernews (hackernews)
  [ ON] arxiv (arxiv)
  [ ON] rss (rss)
  [ ON] substack (substack)
  [ ON] blogs (blog)
  [ ON] exa (exa)
  [ ON] newsletters (newsletter)
  [off] pubmed (pubmed)
  [off] twitter (twitter)
  [off] twitter_accounts (twitter_accounts)
  [off] podcasts (podcast)
  [off] youtube (youtube)


## Step 2: Fetch — Individual Sources

Each fetcher gets its own cell. The config driving it is printed at the top so you can see exactly what produces what.

Results accumulate in `all_items`.

In [3]:
all_items: list[RawItem] = []

def show_fetch(items, label):
    print(f"\n--- {label}: {len(items)} items ---")
    for it in items[:8]:
        has = 'content' if it.raw_content else ('snippet' if it.content_snippet else 'no-text')
        print(f"  {it.title[:75]}")
        print(f"    [{has}] {it.url[:70]}")
    if len(items) > 8:
        print(f"  ... and {len(items) - 8} more")

def show_source_config(key):
    """Print the config for a source from the CONFIG dict."""
    cfg = CONFIG["sources"][key]
    print(f"CONFIG['sources']['{key}']:")
    print(json.dumps(cfg, indent=2, default=str))

### Reddit

In [10]:
CONFIG = {
    "name": "AI Agents Weekly",
    "description": "What happened in AI agents this week",

    "interests": {
        "field": "AI agents & agentic systems",
        "topics": [
            "AI agent frameworks",
            "tool use and function calling",
            "MCP (Model Context Protocol)",
            "multi-agent orchestration",
            "agent memory and planning",
            "code generation agents",
            "autonomous coding (Devin, Cursor, Claude Code)",
        ],
        "level": "advanced",
        "depth": "technical",
    },

    "discovery": {
        "timeframe_hours": 168,
        "max_topics": 10,
        "relevance_threshold": 0.3,
        "n_clusters": 12,
    },

    "sources": {
        "reddit": {
            "enabled": True,
            "subreddits": ["MachineLearning", "LocalLLaMA", "ClaudeAI", "ChatGPTPro"],
            "min_score": 100,
            "max_results": 50,
            "hours_back": 168,
        },
        "hackernews": {
            "enabled": True,
            "filter": "AI agents OR MCP OR Claude OR agentic OR multi-agent OR tool use OR function calling",
            "min_points": 20,
            "max_results": 30,
            "hours_back": 168,
        },
        "arxiv": {
            "enabled": True,
            "categories": ["cs.AI", "cs.CL", "cs.SE"],
            "max_results": 30,
            "hours_back": 168,
        },
        "rss": {
            "enabled": True,
            "feeds": {
                "openai": "https://openai.com/blog/rss/",
                "anthropic": "https://www.anthropic.com/feed.xml",
                "google_ai": "https://blog.google/technology/ai/rss/",
                "deepmind": "https://deepmind.google/blog/rss.xml",
            },
            "hours_back": 168,
        },
        "substack": {
            "enabled": True,
            "publications": {
                "import_ai": "https://importai.substack.com",
                "ahead_of_ai": "https://magazine.sebastianraschka.com",
                "interconnects": "https://www.interconnects.ai",
                "latent_space": "https://www.latent.space",
            },
            "hours_back": 168,
        },
        "blogs": {
            "enabled": True,
            "feeds": {
                "simon_willison": "https://simonwillison.net/atom/everything/",
                "huggingface": "https://huggingface.co/blog/feed.xml",
                "langchain": "https://blog.langchain.dev/rss/",
                "llamaindex": "https://www.llamaindex.ai/blog/rss.xml",
            },
            "hours_back": 168,
        },
        "exa": {
            "enabled": True,
            "query": "AI agents frameworks tool use MCP agentic coding",
            "category": "news",
            "max_results": 20,
            "hours_back": 168,
        },
        "newsletters": {
            "enabled": True,
            "newsletters": [
                {"name": "TLDR AI", "type": "tldr", "section": "ai"},
                {"name": "The Batch", "type": "deeplearning_batch"},
            ],
            "hours_back": 168,
        },
        "pubmed": {
            "enabled": False,
            "query": "large language model agents",
            "max_results": 10,
            "hours_back": 168,
        },
        "twitter": {
            "enabled": False,
            "keywords": ["AI agents", "MCP", "Claude Code", "agentic"],
            "lang": "en",
        },
        "twitter_accounts": {
            "enabled": False,
            "accounts": ["AnthropicAI", "OpenAI", "LangChainAI", "llaboratory"],
        },
        "podcasts": {
            "enabled": False,
            "shows": [
                {"name": "Latent Space", "feed_url": None},
                {"name": "Gradient Dissent", "feed_url": None},
            ],
            "hours_back": 168,
        },
        "youtube": {
            "enabled": False,
            "channels": {
                "Two Minute Papers": "UCbfYPyITQ-7l4upoX8nvctg",
                "Yannic Kilcher": "UCZHmQk67mSJgfCCTn7xBfew",
            },
            "max_results": 10,
            "hours_back": 168,
        },
    },
}

# --- Normalize config into the flat profile format the pipeline expects ---
from discovery import _normalize_config

profile = _normalize_config(CONFIG)
source_configs = {s["id"]: s for s in profile["sources"]}

print(f"Config: {CONFIG['name']}")
print(f"Field: {profile['field']}")
print(f"Topics: {profile['focus_areas']}")
print(f"Threshold: {profile['relevance_threshold']}, Clusters: {profile['n_clusters']}")
print(f"\nSources:")
for s in profile["sources"]:
    status = "ON" if s.get("enabled", True) else "off"
    print(f"  [{status:>3}] {s['id']} ({s['type']})")

Config: AI Agents Weekly
Field: AI agents & agentic systems
Topics: ['AI agent frameworks', 'tool use and function calling', 'MCP (Model Context Protocol)', 'multi-agent orchestration', 'agent memory and planning', 'code generation agents', 'autonomous coding (Devin, Cursor, Claude Code)']
Threshold: 0.3, Clusters: 12

Sources:
  [ ON] reddit (reddit)
  [ ON] hackernews (hackernews)
  [ ON] arxiv (arxiv)
  [ ON] rss (rss)
  [ ON] substack (substack)
  [ ON] blogs (blog)
  [ ON] exa (exa)
  [ ON] newsletters (newsletter)
  [off] pubmed (pubmed)
  [off] twitter (twitter)
  [off] twitter_accounts (twitter_accounts)
  [off] podcasts (podcast)
  [off] youtube (youtube)


In [11]:
from fetchers import fetch_reddit

show_source_config("reddit")
reddit_items = fetch_reddit(source_configs["reddit"])
all_items.extend(reddit_items)
show_fetch(reddit_items, "Reddit")

CONFIG['sources']['reddit']:
{
  "enabled": true,
  "subreddits": [
    "MachineLearning",
    "LocalLLaMA",
    "ClaudeAI",
    "ChatGPTPro"
  ],
  "min_score": 100,
  "max_results": 50,
  "hours_back": 168
}

--- Reddit: 106 items ---
  [D] Is Conference prestige slowing reducing?
    [snippet] https://i.redd.it/66xzqzcu95lg1.png
  [D] Papers with no code
    [snippet] https://www.reddit.com/r/MachineLearning/comments/1rdca7x/d_papers_wit
  [D] ML Engineers — How did you actually learn PyTorch? I keep forgetting ev
    [snippet] https://www.reddit.com/r/MachineLearning/comments/1repn7v/d_ml_enginee
  [D] Why do people say that GANs are dead or outdated when they're still com
    [snippet] https://www.reddit.com/r/MachineLearning/comments/1rbgsey/d_why_do_peo
  [D] Is the move toward Energy-Based Models for reasoning a viable exit from
    [snippet] https://www.reddit.com/r/MachineLearning/comments/1rco6go/d_is_the_mov
  Anthropic: "We’ve identified industrial-scale distillation attac

### Hacker News

In [12]:
source_configs["hackernews"] = {'enabled': True,
 'filter': 'AI agents OR MCP OR Claude OR agentic OR multi-agent OR tool use OR function calling',
 'min_points': 100,
 'id': 'hackernews',
 'type': 'hackernews'}

In [6]:
from fetchers import fetch_hackernews

hn_items = fetch_hackernews({'enabled': True,
 'filter': 'AI agents OR MCP OR Claude OR agentic OR multi-agent OR tool use OR function calling',
 'min_points': 100,
 'id': 'hackernews',
 'type': 'hackernews'})
all_items.extend(hn_items)
show_fetch(hn_items, "HackerNews")


--- HackerNews: 1 items ---
  Don't trust AI agents
    [no-text] https://nanoclaw.dev/blog/nanoclaw-security-model


### RSS Feeds

In [ ]:
from fetchers import fetch_rss
show_source_config("rss")
rss_items = fetch_rss(source_configs["rss"])
all_items.extend(rss_items)
show_fetch(rss_items, "RSS")

CONFIG['sources']['rss']:
{
  "enabled": true,
  "feeds": {
    "openai": "https://openai.com/blog/rss/",
    "anthropic": "https://www.anthropic.com/feed.xml",
    "google_ai": "https://blog.google/technology/ai/rss/",
    "deepmind": "https://deepmind.google/blog/rss.xml"
  },
  "hours_back": 168
}

--- RSS: 7 items ---
  Google and the Massachusetts AI Hub are launching a new AI training initiat
    [snippet] https://blog.google/company-news/outreach-and-initiatives/grow-with-go
  Get more context and understand translations more deeply with new AI-powere
    [snippet] https://blog.google/products-and-platforms/products/translate/translat
  Build with Nano Banana 2, our best image generation and editing model
    [snippet] https://blog.google/innovation-and-ai/technology/developers-tools/buil
  Nano Banana 2: Combining Pro capabilities with lightning-fast speed
    [snippet] https://blog.google/innovation-and-ai/technology/ai/nano-banana-2/
  See the whole picture and find the look 

### arXiv

In [ ]:
from fetchers import fetch_arxiv
show_source_config("arxiv")
arxiv_items = fetch_arxiv(source_configs["arxiv"])
all_items.extend(arxiv_items)
show_fetch(arxiv_items, "arXiv")

CONFIG['sources']['arxiv']:
{
  "enabled": true,
  "categories": [
    "cs.AI"
  ],
  "max_results": 30,
  "hours_back": 168
}

--- arXiv: 30 items ---
  Model Agreement via Anchoring
    [snippet] https://arxiv.org/abs/2602.23360v1
  SeeThrough3D: Occlusion Aware 3D Control in Text-to-Image Generation
    [snippet] https://arxiv.org/abs/2602.23359v1
  SOTAlign: Semi-Supervised Alignment of Unimodal Vision and Language Models 
    [snippet] https://arxiv.org/abs/2602.23353v1
  FlashOptim: Optimizers for Memory Efficient Training
    [snippet] https://arxiv.org/abs/2602.23349v1
  Understanding Usage and Engagement in AI-Powered Scientific Research Tools:
    [snippet] https://arxiv.org/abs/2602.23335v1
  Bitwise Systolic Array Architecture for Runtime-Reconfigurable Multi-precis
    [snippet] https://arxiv.org/abs/2602.23334v1
  Utilizing LLMs for Industrial Process Automation
    [snippet] https://arxiv.org/abs/2602.23331v1
  Toward Expert Investment Teams:A Multi-Agent LLM System with

### Substack

In [ ]:
from fetchers import fetch_substack

show_source_config("substack")
substack_items = fetch_substack(source_configs["substack"])
all_items.extend(substack_items)
show_fetch(substack_items, "Substack")

### Blogs

In [ ]:
from fetchers import fetch_blog

show_source_config("blogs")
blog_items = fetch_blog(source_configs["blogs"])
all_items.extend(blog_items)
show_fetch(blog_items, "Blogs")

### Exa

In [ ]:
from fetchers import fetch_exa

show_source_config("exa")
exa_items = fetch_exa(source_configs["exa"])
all_items.extend(exa_items)
show_fetch(exa_items, "Exa")

### Newsletters (The Batch, TLDR)

In [ ]:
from fetchers import fetch_newsletter

show_source_config("newsletters")
newsletter_items = fetch_newsletter(source_configs["newsletters"])
all_items.extend(newsletter_items)
show_fetch(newsletter_items, "Newsletters")

### PubMed (disabled by default)

In [ ]:
from fetchers import fetch_pubmed

show_source_config("pubmed")
pubmed_items = fetch_pubmed(source_configs["pubmed"])
all_items.extend(pubmed_items)
show_fetch(pubmed_items, "PubMed")

### Twitter keywords (disabled — needs API key)

In [ ]:
from fetchers import fetch_twitter

show_source_config("twitter")
twitter_items = fetch_twitter(source_configs["twitter"])
all_items.extend(twitter_items)
show_fetch(twitter_items, "Twitter keywords")

### Twitter accounts (disabled — needs API key)

In [ ]:
from fetchers import fetch_twitter_accounts

show_source_config("twitter_accounts")
twitter_acc_items = fetch_twitter_accounts(source_configs["twitter_accounts"])
all_items.extend(twitter_acc_items)
show_fetch(twitter_acc_items, "Twitter accounts")

### Podcasts (disabled by default)

In [ ]:
from fetchers import fetch_podcast

show_source_config("podcasts")
podcast_items = fetch_podcast(source_configs["podcasts"])
all_items.extend(podcast_items)
show_fetch(podcast_items, "Podcasts")

### YouTube (disabled by default)

In [ ]:
from fetchers import fetch_youtube

show_source_config("youtube")
yt_items = fetch_youtube(source_configs["youtube"])
all_items.extend(yt_items)
show_fetch(yt_items, "YouTube")

### Fetch Summary

In [ ]:
source_counts = Counter(item.source_type for item in all_items)
print(f"=== Total: {len(all_items)} items ===\n")
for src, count in source_counts.most_common():
    print(f"  {src:20s} {count:3d}")

raw_items = list(all_items)

## Step 3: Extract Content

In [ ]:
from discovery import extract_content_batch

before_count = sum(1 for it in raw_items if it.raw_content)
raw_items = extract_content_batch(raw_items)
after_count = sum(1 for it in raw_items if it.raw_content)

print(f"Content before: {before_count}/{len(raw_items)}")
print(f"Content after:  {after_count}/{len(raw_items)}")

## Step 4: Embed + Dedup + Cluster

In [ ]:
from embedding import run_embedding_pipeline

N_CLUSTERS = profile.get("n_clusters", 12)

topics_dir = str(Path("data/topics").resolve())
data_dir = str(Path("data").resolve())

clusters = run_embedding_pipeline(
    raw_items, profile, topics_dir, data_dir=data_dir, n_clusters=N_CLUSTERS
)

print(f"\n=== {len(clusters)} clusters ===")
for c in clusters:
    print(f"  [{c.cluster_id:2d}] ({len(c.items):2d} items) {c.representative_title[:80]}")

In [ ]:
# Inspect a specific cluster
CLUSTER_IDX = 0

c = clusters[CLUSTER_IDX]
print(f"Cluster {c.cluster_id}: {c.representative_title}")
print(f"Items: {len(c.items)}")
for item in c.items:
    print(f"  - [{item.source_type}] {item.title}")
    if item.content_snippet:
        print(f"    {item.content_snippet[:120]}...")

## Step 5: LLM Ranking

Edit the ranking prompt and threshold below, then run to see scores.

In [ ]:
from openai import OpenAI

MINIMAX_API_KEY = os.getenv("MINIMAX_API_KEY", "")
client = OpenAI(api_key=MINIMAX_API_KEY, base_url="https://api.minimax.io/v1")
MODEL = "MiniMax-M2.5"

RELEVANCE_THRESHOLD = profile.get("relevance_threshold", 0.5)

def strip_think(text):
    return re.sub(r"<think>.*?</think>\s*", "", text, flags=re.DOTALL).strip()

field = profile.get("field", "")
focus_areas = ", ".join(profile.get("focus_areas", []))
level = profile.get("level", "advanced")

cluster_summaries = []
for i, c in enumerate(clusters):
    titles = [it.title for it in c.items[:5]]
    snippets = [it.content_snippet[:150] for it in c.items[:3] if it.content_snippet]
    cluster_summaries.append(
        f"[{i}] {c.representative_title}\n"
        f"  Sources: {len(c.items)} items\n"
        f"  Titles: {'; '.join(titles)}\n"
        f"  Snippets: {' | '.join(snippets)}"
    )

print(f"Threshold: {RELEVANCE_THRESHOLD}")
print(f"\n=== Cluster summaries sent to LLM ===")
for s in cluster_summaries:
    print(s)
    print()

In [ ]:
# ============================================================
# RANKING PROMPT — edit this to change how topics are scored
# ============================================================

RANKING_PROMPT = f"""You are a content relevance scorer for a personalized news feed.

User profile:
- Field: {field}
- Focus areas: {focus_areas}
- Level: {level}

Score each topic cluster 0.0-1.0 for relevance to this user. Also provide a one-sentence reason.

Topics to score:
{chr(10).join(cluster_summaries)}

Respond with ONLY a JSON array. Each element: {{"index": N, "score": 0.0-1.0, "reason": "..."}}.
No other text."""

print(f"Prompt length: {len(RANKING_PROMPT)} chars")

In [ ]:
# Call the LLM for ranking
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=4096,
    messages=[{"role": "user", "content": RANKING_PROMPT}],
)

raw_text = response.choices[0].message.content
text = strip_think(raw_text)

if text.startswith("```"):
    text = re.sub(r"^```\w*\n?", "", text)
    text = re.sub(r"\n?```$", "", text)

scores = json.loads(text)

print(f"=== LLM Scores (threshold={RELEVANCE_THRESHOLD}) ===\n")
for entry in sorted(scores, key=lambda e: e["score"], reverse=True):
    idx = entry["index"]
    score = entry["score"]
    reason = entry.get("reason", "")
    title = clusters[idx].representative_title if idx < len(clusters) else "?"
    marker = ">>>" if score >= RELEVANCE_THRESHOLD else "   "
    print(f"  {marker} [{idx:2d}] {score:.2f}  {title[:60]}")
    print(f"         {reason}")

In [ ]:
# Apply scores + filter
for entry in scores:
    idx = entry["index"]
    if 0 <= idx < len(clusters):
        clusters[idx].relevance_score = entry["score"]
        clusters[idx].relevance_reason = entry.get("reason", "")

ranked = [c for c in clusters if c.relevance_score >= RELEVANCE_THRESHOLD]
ranked.sort(key=lambda c: c.relevance_score, reverse=True)

print(f"\n{len(ranked)} topics above threshold {RELEVANCE_THRESHOLD}:")
for c in ranked:
    print(f"  {c.relevance_score:.2f}  {c.representative_title}")

## Step 6: LLM Synthesis

Edit the synthesis prompt below, then run to generate topic summaries.

In [ ]:
TOPIC_IDX = 0

cluster = ranked[TOPIC_IDX]
depth = profile.get("depth", "technical")

source_texts = []
for item in cluster.items[:6]:
    parts = [f"Source ({item.source_type}): {item.title}"]
    if item.raw_content:
        parts.append(item.raw_content[:1500])
    elif item.content_snippet:
        parts.append(item.content_snippet[:500])
    source_texts.append("\n".join(parts))

print(f"Topic: {cluster.representative_title}")
print(f"Sources: {len(cluster.items)} items ({len(source_texts)} used for synthesis)")
for st in source_texts:
    print(f"  - {st.split(chr(10))[0]}")

In [ ]:
# ============================================================
# SYNTHESIS PROMPT — edit this to change how topics are written
# ============================================================

SYNTHESIS_PROMPT = f"""Write a concise synthesis of this topic for a personalized learning feed.

Topic: {cluster.representative_title}
User level: {level}
Depth: {depth}

Source material:
{chr(10).join(source_texts)}

Guidelines:
- Start with a one-paragraph TL;DR
- Then 2-3 key insights or developments
- Adapt language to the user's level ({level}) and depth ({depth})
- If sources conflict, note the disagreement
- End with "Why this matters" (one sentence)
- Use markdown formatting
- Keep it under 400 words"""

print(f"Prompt length: {len(SYNTHESIS_PROMPT)} chars")

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": SYNTHESIS_PROMPT}],
)

synthesis = strip_think(response.choices[0].message.content)

from IPython.display import Markdown, display
display(Markdown(synthesis))

## Step 7: Synthesize All + Write Folders

In [ ]:
from discovery import write_topic_folder

results = []
for i, cluster in enumerate(ranked):
    print(f"\n[{i+1}/{len(ranked)}] Synthesizing: {cluster.representative_title}...")
    
    src_texts = []
    for item in cluster.items[:6]:
        parts = [f"Source ({item.source_type}): {item.title}"]
        if item.raw_content:
            parts.append(item.raw_content[:1500])
        elif item.content_snippet:
            parts.append(item.content_snippet[:500])
        src_texts.append("\n".join(parts))
    
    prompt = f"""Write a concise synthesis of this topic for a personalized learning feed.

Topic: {cluster.representative_title}
User level: {level}
Depth: {depth}

Source material:
{chr(10).join(src_texts)}

Guidelines:
- Start with a one-paragraph TL;DR
- Then 2-3 key insights or developments
- Adapt language to the user's level ({level}) and depth ({depth})
- If sources conflict, note the disagreement
- End with "Why this matters" (one sentence)
- Use markdown formatting
- Keep it under 400 words"""

    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    syn = strip_think(resp.choices[0].message.content)
    
    slug = write_topic_folder(cluster, syn, profile)
    if slug:
        print(f"  -> wrote: data/topics/{slug}/")
        results.append(slug)
    else:
        print(f"  -> skipped (already exists)")

print(f"\n=== Done: {len(results)} topic folders created ===")

## Inspect Output

In [ ]:
import yaml
topics_path = Path("data/topics")
for folder in sorted(topics_path.iterdir()):
    if folder.is_dir():
        meta_file = folder / "meta.yaml"
        if meta_file.exists():
            meta = yaml.safe_load(meta_file.read_text())
            score = meta.get("relevance_score", 0)
            n_sources = len(meta.get("sources", []))
            print(f"  {score:.2f}  {meta['title'][:60]}  ({n_sources} sources, {meta.get('category','?')})")

In [ ]:
TOPIC_SLUG = ""  # leave empty for latest

if not TOPIC_SLUG:
    folders = sorted(topics_path.iterdir())
    TOPIC_SLUG = folders[-1].name if folders else ""

syn_path = topics_path / TOPIC_SLUG / "synthesis.md"
if syn_path.exists():
    display(Markdown(f"**Folder:** `{TOPIC_SLUG}`\n\n---\n\n" + syn_path.read_text()))
else:
    print(f"Not found: {syn_path}")